# TerraPulse AI · 01: build the fine-tuning data (Colab)
Open this notebook in VS Code with the **Colab extension** and pick a **T4 GPU** runtime. Nothing here runs on your laptop.

1. intent data: generated with exact labels (`training/gen_intent_data.py`)
2. real fact sheets: the live pipeline (SLM off) on ~160 generated questions (`training/build_compose_data.py`)
3. answer drafts: an **open** teacher (Qwen3-8B, 4-bit) writes answers from each fact sheet; only drafts that pass
   the runtime **grounding verifier** are kept. No hosted/proprietary LLM is used anywhere.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Get the project into the Colab runtime. Option A: clone your GitHub repo. Option B: upload GEOAI.zip.
REPO_URL = "https://github.com/aaron43210/GeoAI_Hackathon.git"   # set "" to upload GEOAI.zip instead
import os, subprocess, sys
if REPO_URL and os.path.exists("/content/GEOAI/.git"):
    subprocess.run(["git", "-C", "/content/GEOAI", "pull", "--ff-only"], check=True)
elif REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/GEOAI"], check=True)
elif not os.path.exists("/content/GEOAI"):
    from google.colab import files
    up = files.upload()                         # choose GEOAI.zip (project folder zipped, without node_modules)
    subprocess.run(["unzip", "-q", next(iter(up)), "-d", "/content/"], check=True)
%cd /content/GEOAI
!pip -q install -r backend/requirements-core.txt
sys.path.insert(0, "/content/GEOAI/backend")

In [ ]:
!python training/gen_intent_data.py --n 3000 --out training/data/intent.jsonl
!head -c 600 training/data/intent.jsonl

### Real fact sheets from live APIs (≈15–25 min, paced to respect free-API limits)

In [ ]:
!python training/build_compose_data.py --n 160 --pause 2 --out training/data/factsheets.jsonl

### Open teacher drafts → keep only answers that pass the grounding verifier

In [ ]:
!pip -q install -U "transformers>=4.51" accelerate bitsandbytes
import json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from app.slm.prompts import compose_messages
from app.evidence.grounding import verify

TEACHER = "Qwen/Qwen3-8B"          # Apache-2.0 open weights
tok = AutoTokenizer.from_pretrained(TEACHER)
teacher = AutoModelForCausalLM.from_pretrained(TEACHER, device_map="auto",
            quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16))

def draft(sheet, temperature=0.6):
    msgs = compose_messages(sheet)
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    ids = tok(text, return_tensors="pt").to(teacher.device)
    out = teacher.generate(**ids, max_new_tokens=420, do_sample=True, temperature=temperature, top_p=0.9)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
rows = [json.loads(l) for l in open("training/data/factsheets.jsonl")]
kept, stats = [], {"teacher": 0, "template": 0, "dropped": 0}
for i, r in enumerate(rows):
    best = None
    for t in (0.5, 0.8):
        a = draft(r["fact_sheet"], t)
        g = verify(a, r["fact_sheet"], set(r["evidence_ids"]), r["known_names"], r["must_mention"], r["recommended"], r.get("evidence_values"))
        if g.status == "passed":
            best = ("teacher", a); break
    if best is None and stats["template"] < 0.25 * len(rows):
        best = ("template", r["template_answer"])        # grounded by construction; capped so style stays natural
    if best is None:
        stats["dropped"] += 1; continue
    stats[best[0]] += 1
    msgs = compose_messages(r["fact_sheet"]) + [{"role": "assistant", "content": best[1]}]
    kept.append({"task": "compose", "source": best[0], "messages": msgs})
    if i % 10 == 0: print(i, stats)
with open("training/data/compose.jsonl", "w") as fh:
    for k in kept: fh.write(json.dumps(k, ensure_ascii=False) + "\n")
print("final", stats, "kept", len(kept))

In [ ]:
# Hugging Face login: uses the Colab secret HF_TOKEN if available, otherwise asks (input is hidden, never saved).
import os
from huggingface_hub import login
token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
if not token:
    from getpass import getpass
    token = getpass("Hugging Face write token: ")
login(token)
os.environ["HF_TOKEN"] = token
HF_USER = "aaronr43210"

In [ ]:
# Optional: publish the datasets (open licence) so the model card can link them
from huggingface_hub import HfApi
api = HfApi(); repo = f"{HF_USER}/terrapulse-sft-data"
api.create_repo(repo, repo_type="dataset", exist_ok=True)
for f in ["intent.jsonl", "compose.jsonl", "factsheets.jsonl"]:
    api.upload_file(path_or_fileobj=f"training/data/{f}", path_in_repo=f, repo_id=repo, repo_type="dataset")
print("https://huggingface.co/datasets/" + repo)